# cuda-empty-cache — ex2: torch.cuda.empty_cache() is a safe no-op on CPU-only — exercise the real call

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `cuda-empty-cache`. Running the final beacon cell reports progress against the `PyTorch: torch.cuda.empty_cache` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: torch.cuda.empty_cache` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`cuda-empty-cache`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "cuda-empty-cache"
DD_SUBTOPIC = "PyTorch: torch.cuda.empty_cache"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `torch.cuda.empty_cache()` on CPU-only — it's a safe no-op

Ex1 patched `empty_cache` with a mock to count calls. The deepening move: on a CPU-only machine (no GPU), `torch.cuda.empty_cache()` IS A NO-OP. It does not raise. You can ship a single code path that calls it unconditionally.

```python
torch.cuda.empty_cache()       # CPU-only: returns None, no error
torch.cuda.is_available()      # CPU-only: False
```

**Why this matters in practice.** Training code with GPU instrumentation should still run on the CI box and the laptop. Wrapping every CUDA call in `if cuda.is_available():` clutters the code; relying on the silent no-op (where PyTorch provides one) keeps the call sites clean.

**Not ALL CUDA APIs are no-ops on CPU.** `cuda.synchronize()` also no-ops on CPU. But `torch.tensor([1]).cuda()` raises `AssertionError` because there's no device to send to. The is-available check is mandatory for OPS that require GPU memory; optional for ones that PyTorch made degenerate-safe.

**Drill semantics.** This deepening drill walks the CPU code path explicitly: assert `is_available()` returns False (on the runner), then call `empty_cache()` and confirm no exception. The mock approach from ex1 is gone — we exercise the REAL function.

### Exercise 2 — torch.cuda.empty_cache() is a safe no-op on CPU-only — exercise the real call

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `torch.cuda.is_available()` as a runtime gate, then demonstrate that `torch.cuda.empty_cache()` does not raise on a CPU-only machine — making a single unconditional cleanup call portable across hardware.
> Keywords: cuda, cpu, no-op, is_available
> ```

**KCs targeted:** `is-available-cpu-false`, `empty-cache-no-raise-on-cpu`

Implement `ex2_cpu_safe_empty_cache(n_calls)`.

Unlike ex1 (which patched the function with a mock), this drill calls the REAL `torch.cuda.empty_cache()` and asserts it doesn't raise.

Steps:
1. Read `is_avail = torch.cuda.is_available()`.
2. Call `torch.cuda.empty_cache()` exactly `n_calls` times in a loop. Wrap each call in a `try/except Exception as e:` and record whether ANY call raised.
3. Return:
   ```
   {
     'cuda_is_available': bool,
     'n_calls_attempted': int,
     'any_raised': bool,
     'first_error': str  ('' if none),
   }
   ```

Constraints:
- DO NOT mock `torch.cuda.empty_cache`. Call the real function.
- DO NOT branch on `is_available` to skip the call — the whole point is that the unconditional call works.
- DO NOT call any other CUDA API (no `cuda.synchronize`, no `.cuda()` on tensors) — only `empty_cache` is guaranteed to no-op on CPU.

Drill semantics: on a CPU-only runner, `is_avail` is False, `any_raised` is False — i.e. the call is silent.

In [ ]:
def ex2_cpu_safe_empty_cache(n_calls: int) -> dict:
    """Call torch.cuda.empty_cache() unconditionally; report CPU-only safety."""
    raise NotImplementedError()


def _test_ex2():
    import torch

    # === Single call: must not raise on CPU-only ===
    rep = ex2_cpu_safe_empty_cache(n_calls=1)
    assert isinstance(rep['cuda_is_available'], bool), f'cuda_is_available must be bool, got {type(rep["cuda_is_available"]).__name__}'
    assert rep['n_calls_attempted'] == 1, f'expected 1 call, got {rep["n_calls_attempted"]}'
    assert rep['any_raised'] is False, f'empty_cache must not raise on CPU; first_error={rep["first_error"]!r}'
    assert rep['first_error'] == '', f'no error string when nothing raised, got {rep["first_error"]!r}'

    # === Many calls in a row: still no raise ===
    rep = ex2_cpu_safe_empty_cache(n_calls=50)
    assert rep['n_calls_attempted'] == 50
    assert rep['any_raised'] is False, f'50 calls failed: {rep["first_error"]!r}'

    # === n_calls=0 → no calls, trivially no raise ===
    rep = ex2_cpu_safe_empty_cache(n_calls=0)
    assert rep['n_calls_attempted'] == 0
    assert rep['any_raised'] is False

    # === cuda_is_available is consistent with torch.cuda.is_available() ===
    rep = ex2_cpu_safe_empty_cache(n_calls=1)
    assert rep['cuda_is_available'] == torch.cuda.is_available(), 'reported is_available must match torch.cuda.is_available()'

    # === The function did NOT patch torch.cuda.empty_cache ===
    # (the real function should still be importable and unmocked)
    import inspect
    # Either it's a function/builtin (real) or a wrapper; just ensure it's not a MagicMock
    from unittest.mock import MagicMock
    assert not isinstance(torch.cuda.empty_cache, MagicMock), 'must not leave a mock installed on torch.cuda.empty_cache'

    # === Return shape is exactly the documented keys ===
    required = {'cuda_is_available', 'n_calls_attempted', 'any_raised', 'first_error'}
    assert set(rep.keys()) >= required, f'missing required keys: {required - set(rep.keys())}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_cpu_safe_empty_cache(n_calls):
    import torch
    is_avail = torch.cuda.is_available()
    any_raised = False
    first_error = ''
    for _ in range(n_calls):
        try:
            torch.cuda.empty_cache()
        except Exception as e:
            if not any_raised:
                first_error = repr(e)
            any_raised = True
    return {
        'cuda_is_available': is_avail,
        'n_calls_attempted': n_calls,
        'any_raised': any_raised,
        'first_error': first_error,
    }
```

**`torch.cuda.empty_cache()` is degenerate-safe on CPU by design.** It releases unused blocks held by the caching allocator; on a host with no CUDA device, the allocator has nothing to release, so the function returns immediately. This is a contract — not a happy accident.

**Why not wrap in `if cuda.is_available():`.** You could. But the noisy wrap repeats at every call site. Trust the no-op for APIs where PyTorch guarantees it. For ops that DO require GPU (`.cuda()`, `cuda.synchronize()` is also safe actually, but tensor.cuda() is not), explicit gating is mandatory.

**`torch.cuda.is_available()` reports hardware, not import.** `import torch` works regardless of CUDA presence — only the operations that need a device fail. `is_available()` is the load-bearing runtime check.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()